# 02 — Execução dos Experimentos (com Retomada Automática e ETA)

Execute antes o `00_setup_colab.ipynb` (para montar o Drive e definir `TCC_RESULTS_DIR`).
Se a sessão do Colab cair, basta rodar novamente este notebook: todo run que já tiver `metrics.json` será pulado automaticamente.


In [ ]:
# === 1. Smoke test por arquitetura real (1 passo, batch=2) ===
# Valida compatibilidade de memória (VRAM), bf16/fp16, ModernBERT e BigBird antes da grade principal.
!python -m tcc.runner --smoke-test

In [ ]:
# === 2. Dry-run rápido (validação ponta a ponta do pipeline) ===
!python -m tcc.runner --config configs/experiments/dry_run.yaml

In [ ]:
# === 3. Grade principal (70 runs: 5 modelos @ 512 + 2 longos @ 2048 × 10 seeds) ===
# Dica: você pode rodar por subconjuntos com --models ou --seeds se quiser sessões menores.
!python -m tcc.runner --config configs/experiments/main.yaml

In [ ]:
# === 4. Empacotar results/ em .zip para download rápido ===
import os
from pathlib import Path
from tcc.config import load_config
from tcc.utils import zip_results_dir

cfg = load_config()
zip_path = zip_results_dir(cfg["paths"]["results_dir"], Path("/content/results_tcc.zip") if Path("/content").exists() else Path("results_tcc.zip"))
print("Arquivo gerado:", zip_path)

if "COLAB_GPU" in os.environ:
    from google.colab import files
    files.download(str(zip_path))